# 02 — Compare MODIS and VIIRS, raw vs. harmonized (Aug–Oct 2019)

Real data: FIRMS SP archive (MODIS Terra+Aqua, VIIRS S-NPP, VIIRS NOAA-20) over `config.INDONESIA_BBOX`
(not clipped to Indonesia). Method: `harmonization.harmonize`, see `lode/harmonization/cell-day-method.md`:
0.01° cells, WIB dates, FIRMS `type == 0` only, all confidence levels kept.

**Questions**
1. What does harmonization exclude, and how much?
2. Does the VIIRS/MODIS ratio become more stable with active cell-days than with raw detection counts?
3. How often do the sensors agree on a cell-day?
4. How much does NOAA-20 add on top of S-NPP?
5. How much activity rests on low-confidence detections only?

In [1]:
import pandas as pd
import plotly.express as px
import datetime as dt
from eyes_on_karhutla import activity, harmonization, ingestion, preprocessing

SOURCES = ['MODIS_SP', 'VIIRS_SNPP_SP', 'VIIRS_NOAA20_SP']
raw = pd.concat([ingestion.read_firms_csv(ingestion.latest_raw(s), s) for s in SOURCES], ignore_index=True)
obs, norm_excluded = preprocessing.normalize(raw)
kept, excluded = harmonization.prepare(obs)
cell_days = harmonization.to_cell_days(kept)
assert len(kept) + len(excluded) == len(obs)
assert sum(map(len, cell_days['source_record_ids'])) == len(kept)
len(raw), len(obs), len(kept), len(cell_days)

(799948, 799948, 780536, 344675)

## 1. Exclusions

In [2]:
pd.crosstab(excluded['source'], excluded['exclusion_reason'], margins=True)

exclusion_reason,non_vegetation_type,All
source,,
MODIS_SP,2532,2532
VIIRS_NOAA20_SP,10037,10037
VIIRS_SNPP_SP,6843,6843
All,19412,19412


In [3]:
(excluded.groupby('source').size() / obs.groupby('source').size()).round(4).rename('excluded_share')

source
MODIS_SP           0.0335
VIIRS_NOAA20_SP    0.0277
VIIRS_SNPP_SP      0.0189
Name: excluded_share, dtype: float64

## 2. Daily VIIRS/MODIS ratio: raw detections vs. active cell-days

Raw = S-NPP detections / MODIS detections (as in notebook 01). Cell-days = VIIRS-hit / MODIS-hit cell-days.

The pull is bounded by UTC dates, so the first WIB date (2019-08-01, missing 00:00–07:00 WIB incl. the night pass)
and the last (2019-11-01, only 00:00–07:00 WIB) are **partial** and are dropped here.

In [4]:
wib_dates = sorted(cell_days['wib_date'].unique())
complete = wib_dates[1:-1]
raw_daily = (kept[kept.source.isin(['MODIS_SP', 'VIIRS_SNPP_SP'])]
             .groupby(['wib_date', 'source']).size().unstack())
cd_daily = cell_days.groupby('wib_date')[['modis_hit', 'viirs_hit']].sum()
cd_daily['either'] = cell_days.groupby('wib_date').size()
raw_daily, cd_daily = raw_daily.loc[complete], cd_daily.loc[complete]
ratios = pd.DataFrame({
    'raw_snpp_over_modis': raw_daily['VIIRS_SNPP_SP'] / raw_daily['MODIS_SP'],
    'celldays_viirs_over_modis': cd_daily['viirs_hit'] / cd_daily['modis_hit'],
})
summary = ratios.describe().T[['count', 'min', '50%', 'max']]
summary['cv'] = ratios.std() / ratios.mean()
summary.round(2)

,count,min,50%,max,cv
raw_snpp_over_modis,91.0,1.70,5.32,10.34,0.29
celldays_viirs_over_modis,91.0,2.36,5.60,12.24,0.33


In [5]:
px.line(ratios.reset_index().melt(id_vars='wib_date', var_name='metric', value_name='ratio'),
        x='wib_date', y='ratio', color='metric', title='Daily VIIRS / MODIS ratio (WIB dates)')

In [6]:
px.line(cd_daily.reset_index().melt(id_vars='wib_date', var_name='series', value_name='active_cell_days'),
        x='wib_date', y='active_cell_days', color='series',
        title='Active cell-days per WIB date (0.01° cells, type 0, all confidence)')

## 2b. Grid-size sensitivity

Does a coarser grid make the ratio more stable or improve agreement?

In [7]:
rows = []
for g in [0.01, 0.02, 0.05, 0.1]:
    k, _ = harmonization.prepare(obs, grid_deg=g)
    cd = harmonization.to_cell_days(k, g)
    d = cd.groupby('wib_date')[['modis_hit', 'viirs_hit']].sum().loc[complete]
    r = d['viirs_hit'] / d['modis_hit']
    rows.append({'grid_deg': g, 'cell_days': len(cd), 'ratio_median': r.median(), 'ratio_min': r.min(),
                 'ratio_max': r.max(), 'ratio_cv': r.std() / r.mean(),
                 'modis_seen_by_viirs': (cd.modis_hit & cd.viirs_hit).sum() / cd.modis_hit.sum()})
pd.DataFrame(rows).round(3)

,grid_deg,cell_days,ratio_median,ratio_min,ratio_max,ratio_cv,modis_seen_by_viirs
0,0.01,344675,5.598,2.359,12.236,0.329,0.574
1,0.02,259540,5.531,2.412,12.460,0.331,0.676
2,0.05,185564,5.469,2.401,11.539,0.317,0.771
3,0.10,136037,4.909,2.425,10.012,0.295,0.827


## 3. Sensor agreement on cell-days

In [8]:
agreement = pd.Series(pd.NA, index=cell_days.index, dtype='string')
agreement[cell_days.modis_hit & cell_days.viirs_hit] = 'both'
agreement[cell_days.modis_hit & ~cell_days.viirs_hit] = 'modis_only'
agreement[~cell_days.modis_hit & cell_days.viirs_hit] = 'viirs_only'
agreement.value_counts(normalize=True).round(3)

viirs_only    0.814
both          0.107
modis_only    0.079
Name: proportion, dtype: double[pyarrow]

In [9]:
# Share of MODIS cell-days also seen by VIIRS on the same WIB date
(cell_days.modis_hit & cell_days.viirs_hit).sum() / cell_days.modis_hit.sum()

np.float64(0.5743653831133329)

In [10]:
# Same, but allowing VIIRS in the 3x3 neighbourhood (geolocation offsets, MODIS pixels up to 4.8 km)
vk = set(zip(kept.loc[kept.sensor == 'VIIRS', 'cell_i'], kept.loc[kept.sensor == 'VIIRS', 'cell_j'],
             kept.loc[kept.sensor == 'VIIRS', 'wib_date']))
m = kept.loc[kept.sensor == 'MODIS', ['cell_i', 'cell_j', 'wib_date']].drop_duplicates()
pd.Series([any((i + a, j + b, d) in vk for a in (-1, 0, 1) for b in (-1, 0, 1))
           for i, j, d in m.itertuples(index=False)]).mean()

np.float64(0.8543153948293368)

## 4. What NOAA-20 adds on top of S-NPP

Counted on VIIRS-only `kept` rows grouped to cell-days per satellite source.

In [11]:
v = kept[kept.sensor == 'VIIRS']
per_src = v.groupby(['cell_i', 'cell_j', 'wib_date'])['source'].agg(set)
pd.Series({
    'viirs_cell_days': len(per_src),
    'snpp_cell_days': per_src.map(lambda s: 'VIIRS_SNPP_SP' in s).sum(),
    'noaa20_only_cell_days': per_src.map(lambda s: s == {'VIIRS_NOAA20_SP'}).sum(),
    'raw_viirs_detections': len(v),
})

viirs_cell_days          317428
snpp_cell_days           195053
noaa20_only_cell_days    122375
raw_viirs_detections     707585
dtype: int64

## 5. Low-confidence-only cell-days

In [12]:
cell_days.groupby(agreement)['low_conf_only'].mean().round(3)

both          0.002
modis_only    0.076
viirs_only    0.078
Name: low_conf_only, dtype: float64

In [13]:
cell_days['low_conf_only'].mean().round(3)

np.float64(0.07)

## 6. Side-by-side series with consistent constellations (`activity`)

`modis` = Terra + Aqua, `viirs_snpp` = S-NPP only (constant since 2012), `viirs_all` adds NOAA-20.
Partial WIB edge dates are flagged by `activity.wib_date_coverage` and left out of period sums.

In [14]:
daily = activity.daily_series(cell_days, dt.date(2019, 8, 1), dt.date(2019, 10, 31), series=tuple(activity.SERIES))
assert daily.loc[daily.complete, 'viirs_all'].equals(cd_daily['viirs_hit'])  # matches section 2
daily[~daily.complete]

,complete,modis,viirs_snpp,viirs_all
wib_date,,,,
2019-08-01,False,286,875,1560
2019-11-01,False,4,171,260


In [15]:
activity.aggregate(daily, 'M')

,modis,viirs_snpp,viirs_all,n_complete_days,n_days
period,,,,,
2019-08,12095,38418,65258,30,31
2019-09,37118,100852,158636,30,30
2019-10,14512,54737,91714,31,31


In [16]:
long = daily[daily.complete].drop(columns='complete').reset_index().melt('wib_date', var_name='series', value_name='active_cell_days')
long['series'] = long['series'].map({k: v['label'] for k, v in activity.SERIES.items()})
px.line(long, x='wib_date', y='active_cell_days', color='series',
        title='Active cell-days per WIB date — sensor series side by side (not summed)')

In [17]:
# How much S-NPP-only loses vs. all VIIRS satellites, per month
m = activity.aggregate(daily, 'M')
(m['viirs_snpp'] / m['viirs_all']).round(3).rename('snpp_share_of_viirs_all')

period
2019-08    0.589
2019-09    0.636
2019-10    0.597
Freq: M, Name: snpp_share_of_viirs_all, dtype: float64

## Conclusions (2019 sample, Indonesia bbox, not clipped)

Real data, but one fire season (an extreme one) and one region, so treat these as preliminary.

1. **Exclusions.** `type != 0` removes 3.4% of MODIS, 1.9% of S-NPP and 2.8% of NOAA-20 detections.
   Nothing else is excluded.
2. **Cell-days do not, by themselves, harmonize the sensors.** After dropping the partial edge dates, VIIRS
   still has about 5.5× as many active cell-days as MODIS (daily range 2.4–12.2, CV 0.33), no more stable
   than raw counts (5.3×, 1.7–10.3, CV 0.29). Coarser grids (up to 0.1°) don't fix it. So the VIIRS excess
   is mainly **detection sensitivity** (VIIRS catches smaller and cooler fires MODIS misses), not pixel
   count. Day-to-day shape still agrees well (daily correlation about 0.92).
3. **Agreement depends on the grid.** 57% of MODIS cell-days have a VIIRS hit in the same 0.01° cell, 85%
   within the 3×3 neighbourhood, and 83% at 0.1°. At 0.01° the same fire often falls into neighbouring
   cells.
4. **S-NPP and NOAA-20 are not redundant per cell.** NOAA-20 alone accounts for 39% of VIIRS cell-days,
   even though their *daily totals* correlate at 0.98. Combining them into `viirs_hit` means the VIIRS
   series gains a second satellite from 2018, which breaks consistency with 2012–2017 (S-NPP only).
5. **Low-confidence-only** cell-days are 7% overall and almost never occur when both sensors agree (0.2%).

6. **S-NPP only gives about 60% of all-VIIRS cell-days** per month (0.59–0.64), so NOAA-20 adds a lot.

**Approach taken.** The cell-day table is a common unit with full provenance and per-platform flags.
Activity is reported as **side-by-side series with fixed constellations**: `modis` (Terra + Aqua) and
`viirs_snpp` (S-NPP only, constant since 2012). They are never summed. `viirs_all` is available but
jumps in 2018. A calibration on the overlap years (method Option 2) is deferred until a longer history
is pulled.